In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import csv
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from spellchecker import SpellChecker
from functools import lru_cache
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import vstack
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score

spell = SpellChecker(distance=1)

# Загружаем пакеты
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /home/charmaip/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/charmaip/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/charmaip/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/charmaip/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
def load_tweets_simple(filename):
    tweets = []

    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)

        for row in reader:
            for tweet in row:
                tweet = tweet.strip()

                if tweet:
                    tweets.append(tweet)

    return pd.DataFrame({'text': tweets})

In [3]:
positive = load_tweets_simple('../datasets/processedPositive.csv')
negative = load_tweets_simple('../datasets/processedNegative.csv')
neutral = load_tweets_simple('../datasets/processedNeutral.csv')

positive['label'] = 2
negative['label'] = 0
neutral['label'] = 1

df = pd.concat(
    [positive, negative, neutral],
    ignore_index=True
)

print(df['label'].value_counts())
print(f"Всего твитов: {len(df)}")
print("\nПримеры твитов:")
print(df.head(10))

label
1    1566
2    1183
0    1116
Name: count, dtype: int64
Всего твитов: 3865

Примеры твитов:
                                                text  label
0             An inspiration in all aspects: Fashion      2
1                                            fitness      2
2    beauty and personality. :)KISSES TheFashionIcon      2
3  Apka Apna Awam Ka Channel Frankline Tv Aam Adm...      2
4  Beautiful album from  the greatest unsung guit...      2
5  Good luck to Rich riding for great project in ...      2
6            Omg he... kissed... him crying with joy      2
7     happy anniv ming and papi!!!!! love love happy      2
8                                       thanks happy      2
9                                       C'mon Tweeps      2


### **Data Preparation**

Делим на x/y, train/test, preprocessing и векторизация

In [4]:
X = df['text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Размеры:")
print("Train:", len(X_train))
print("Test: ", len(X_test))

print("\nРаспределение классов в Train:")
print(y_train.value_counts(normalize=True).round(3))

print("\nРаспределение классов в Test:")
print(y_test.value_counts(normalize=True).round(3))

Размеры:
Train: 3092
Test:  773

Распределение классов в Train:
label
1    0.405
2    0.306
0    0.289
Name: proportion, dtype: float64

Распределение классов в Test:
label
1    0.405
2    0.307
0    0.288
Name: proportion, dtype: float64


In [5]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [6]:
def preprocess(text, method='tokenize'):

    if not isinstance(text, str):
        return ""

    # нижний регистр
    text = text.lower()

    # удаляем ссылки
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)

    # оставляем только английские буквы и пробелы
    text = re.sub(r'[^a-z\s]', ' ', text)

    # убираем лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()

    # разбиваем текст на слова
    tokens = word_tokenize(text)


    # 1. just tokenization
    if method == 'tokenize':
        tokens = tokens


    # 2. stemming
    elif method == 'stem':
        tokens = [
            stemmer.stem(t)
            for t in tokens
        ]


    # 3. lemmatization
    elif method == 'lemma':
        tokens = [
            lemmatizer.lemmatize(t)
            for t in tokens
        ]


    # 4. stemming + stopwords
    elif method == 'stem_stop':
        tokens = [
            stemmer.stem(t)
            for t in tokens
            if t not in stop_words
        ]


    return " ".join(tokens)

In [7]:
example = "I really loved playing with these dogs!!!"

print("Оригинал:")
print(example)

print("\nJust tokenization:")
print(preprocess(example, 'tokenize'))

print("\nStemming:")
print(preprocess(example, 'stem'))

print("\nLemmatization:")
print(preprocess(example, 'lemma'))

print("\nStemming + stopwords:")
print(preprocess(example, 'stem_stop'))

Оригинал:
I really loved playing with these dogs!!!

Just tokenization:
i really loved playing with these dogs

Stemming:
i realli love play with these dog

Lemmatization:
i really loved playing with these dog

Stemming + stopwords:
realli love play dog


In [8]:
# 4 варианта предобработки для train и test

X_train_tok = X_train.apply(
    lambda t: preprocess(t, method='tokenize')
)

X_test_tok = X_test.apply(
    lambda t: preprocess(t, method='tokenize')
)


X_train_stem = X_train.apply(
    lambda t: preprocess(t, method='stem')
)

X_test_stem = X_test.apply(
    lambda t: preprocess(t, method='stem')
)


X_train_lemma = X_train.apply(
    lambda t: preprocess(t, method='lemma')
)

X_test_lemma = X_test.apply(
    lambda t: preprocess(t, method='lemma')
)


X_train_stem_stop = X_train.apply(
    lambda t: preprocess(t, method='stem_stop')
)

X_test_stem_stop = X_test.apply(
    lambda t: preprocess(t, method='stem_stop')
)

In [9]:
print("Исходный твит:")
print(X_train.iloc[0])

print("\nJust tokenization:")
print(X_train_tok.iloc[0])

print("\nStemming:")
print(X_train_stem.iloc[0])

print("\nLemmatization:")
print(X_train_lemma.iloc[0])

print("\nStemming + stopwords:")
print(X_train_stem_stop.iloc[0])

Исходный твит:
Who hacked me unhappy

Just tokenization:
who hacked me unhappy

Stemming:
who hack me unhappi

Lemmatization:
who hacked me unhappy

Stemming + stopwords:
hack unhappi


**Just tokenization**

In [10]:
# Для метода 'tokenize' делаем 3 вида векторизации

print("=== Векторизация tokenize ===")


# 1. Binary
# 1 — слово есть в твите
# 0 — слова нет

bin_vec_tok = CountVectorizer(
    binary=True,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_tok_bin = bin_vec_tok.fit_transform(X_train_tok)
X_test_tok_bin = bin_vec_tok.transform(X_test_tok)

print(f"Binary: {X_train_tok_bin.shape}")


# 2. Counts
# Считаем, сколько раз слово встретилось в твите

cnt_vec_tok = CountVectorizer(
    binary=False,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_tok_cnt = cnt_vec_tok.fit_transform(X_train_tok)
X_test_tok_cnt = cnt_vec_tok.transform(X_test_tok)

print(f"Counts: {X_train_tok_cnt.shape}")


# 3. TF-IDF
# Вес слова зависит от его важности для конкретного текста

tfidf_vec_tok = TfidfVectorizer(
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_tok_tfidf = tfidf_vec_tok.fit_transform(X_train_tok)
X_test_tok_tfidf = tfidf_vec_tok.transform(X_test_tok)

print(f"TF-IDF: {X_train_tok_tfidf.shape}")

=== Векторизация tokenize ===
Binary: (3092, 1096)
Counts: (3092, 1096)
TF-IDF: (3092, 1096)


**Stemming**

In [11]:
# Для метода 'stemming' делаем 3 вида векторизации

print("=== Векторизация stemming ===")


# 1. Binary
bin_vec_stem = CountVectorizer(
    binary=True,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_stem_bin = bin_vec_stem.fit_transform(X_train_stem)
X_test_stem_bin = bin_vec_stem.transform(X_test_stem)

print(f"Binary: {X_train_stem_bin.shape}")


# 2. Counts
cnt_vec_stem = CountVectorizer(
    binary=False,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_stem_cnt = cnt_vec_stem.fit_transform(X_train_stem)
X_test_stem_cnt = cnt_vec_stem.transform(X_test_stem)

print(f"Counts: {X_train_stem_cnt.shape}")


# 3. TF-IDF
tfidf_vec_stem = TfidfVectorizer(
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_stem_tfidf = tfidf_vec_stem.fit_transform(X_train_stem)
X_test_stem_tfidf = tfidf_vec_stem.transform(X_test_stem)

print(f"TF-IDF: {X_train_stem_tfidf.shape}")

=== Векторизация stemming ===
Binary: (3092, 1162)
Counts: (3092, 1162)
TF-IDF: (3092, 1162)


**Lemmatization**

In [12]:
# Для метода 'lemmatization' делаем 3 вида векторизации

print("=== Векторизация lemmatization ===")


# 1. Binary
bin_vec_lemma = CountVectorizer(
    binary=True,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_lemma_bin = bin_vec_lemma.fit_transform(X_train_lemma)
X_test_lemma_bin = bin_vec_lemma.transform(X_test_lemma)

print(f"Binary: {X_train_lemma_bin.shape}")


# 2. Counts
cnt_vec_lemma = CountVectorizer(
    binary=False,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_lemma_cnt = cnt_vec_lemma.fit_transform(X_train_lemma)
X_test_lemma_cnt = cnt_vec_lemma.transform(X_test_lemma)

print(f"Counts: {X_train_lemma_cnt.shape}")


# 3. TF-IDF
tfidf_vec_lemma = TfidfVectorizer(
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_lemma_tfidf = tfidf_vec_lemma.fit_transform(X_train_lemma)
X_test_lemma_tfidf = tfidf_vec_lemma.transform(X_test_lemma)

print(f"TF-IDF: {X_train_lemma_tfidf.shape}")

=== Векторизация lemmatization ===
Binary: (3092, 1112)
Counts: (3092, 1112)
TF-IDF: (3092, 1112)


In [13]:
@lru_cache(maxsize=50000)
def correct_word(word):

    # Очень короткие слова не трогаем
    if len(word) <= 3:
        return word

    # Если слово уже есть в словаре не исправляем
    if word in spell:
        return word

    correction = spell.correction(word)

    # Если spellchecker ничего не нашёл оставляем исходное слово
    if correction is None:
        return word

    return correction

In [14]:
def preprocess_stem_misspellings(text):

    if not isinstance(text, str):
        return ""

    # нижний регистр
    text = text.lower()

    # удаляем ссылки
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)

    # оставляем английские буквы
    text = re.sub(r'[^a-z\s]', ' ', text)

    # убираем лишние пробелы
    text = re.sub(r'\s+', ' ', text).strip()

    # разбиваем на слова
    tokens = word_tokenize(text)

    # сначала исправляем опечатки
    corrected_tokens = [
        correct_word(t)
        for t in tokens
    ]

    # потом применяем stemming
    stemmed_tokens = [
        stemmer.stem(t)
        for t in corrected_tokens
    ]

    return " ".join(stemmed_tokens)

In [15]:
# Stemming + misspellings для train и test

X_train_stem_miss = X_train.apply(
    preprocess_stem_misspellings
)

X_test_stem_miss = X_test.apply(
    preprocess_stem_misspellings
)

print("Готово")
print("Train:", len(X_train_stem_miss))
print("Test:", len(X_test_stem_miss))

print("Исходный твит:")
print(X_train.iloc[0])

print("\nStemming + misspellings:")
print(X_train_stem_miss.iloc[0])

Готово
Train: 3092
Test: 773
Исходный твит:
Who hacked me unhappy

Stemming + misspellings:
who hack me unhappi


**Stemming + misspellings**

In [16]:
# Для метода 'stemming + misspellings' делаем 3 вида векторизации

print("=== Векторизация stemming + misspellings ===")


# 1. Binary
bin_vec_stem_miss = CountVectorizer(
    binary=True,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_stem_miss_bin = bin_vec_stem_miss.fit_transform(X_train_stem_miss)
X_test_stem_miss_bin = bin_vec_stem_miss.transform(X_test_stem_miss)

print(f"Binary: {X_train_stem_miss_bin.shape}")


# 2. Counts
cnt_vec_stem_miss = CountVectorizer(
    binary=False,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_stem_miss_cnt = cnt_vec_stem_miss.fit_transform(X_train_stem_miss)
X_test_stem_miss_cnt = cnt_vec_stem_miss.transform(X_test_stem_miss)

print(f"Counts: {X_train_stem_miss_cnt.shape}")


# 3. TF-IDF
tfidf_vec_stem_miss = TfidfVectorizer(
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_stem_miss_tfidf = tfidf_vec_stem_miss.fit_transform(X_train_stem_miss)
X_test_stem_miss_tfidf = tfidf_vec_stem_miss.transform(X_test_stem_miss)

print(f"TF-IDF: {X_train_stem_miss_tfidf.shape}")

=== Векторизация stemming + misspellings ===
Binary: (3092, 1171)
Counts: (3092, 1171)
TF-IDF: (3092, 1171)


In [17]:
def preprocess_lemma_misspellings(text):

    if not isinstance(text, str):
        return ""

    text = text.lower()

    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)

    text = re.sub(r'[^a-z\s]', ' ', text)

    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)

    corrected_tokens = [
        correct_word(t)
        for t in tokens
    ]

    lemmatized_tokens = [
        lemmatizer.lemmatize(t)
        for t in corrected_tokens
    ]

    return " ".join(lemmatized_tokens)

In [18]:
# Lemmatization + misspellings для train и test

X_train_lemma_miss = X_train.apply(
    preprocess_lemma_misspellings
)

X_test_lemma_miss = X_test.apply(
    preprocess_lemma_misspellings
)

print("Готово")
print("Train:", len(X_train_lemma_miss))
print("Test:", len(X_test_lemma_miss))

Готово
Train: 3092
Test: 773


**Lemmatization + misspellings**

In [19]:
# Для метода 'lemmatization + misspellings' делаем 3 вида векторизации

print("=== Векторизация lemmatization + misspellings ===")


# 1. Binary
bin_vec_lemma_miss = CountVectorizer(
    binary=True,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_lemma_miss_bin = bin_vec_lemma_miss.fit_transform(X_train_lemma_miss)
X_test_lemma_miss_bin = bin_vec_lemma_miss.transform(X_test_lemma_miss)

print(f"Binary: {X_train_lemma_miss_bin.shape}")


# 2. Counts
cnt_vec_lemma_miss = CountVectorizer(
    binary=False,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_lemma_miss_cnt = cnt_vec_lemma_miss.fit_transform(X_train_lemma_miss)
X_test_lemma_miss_cnt = cnt_vec_lemma_miss.transform(X_test_lemma_miss)

print(f"Counts: {X_train_lemma_miss_cnt.shape}")


# 3. TF-IDF
tfidf_vec_lemma_miss = TfidfVectorizer(
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_lemma_miss_tfidf = tfidf_vec_lemma_miss.fit_transform(X_train_lemma_miss)
X_test_lemma_miss_tfidf = tfidf_vec_lemma_miss.transform(X_test_lemma_miss)

print(f"TF-IDF: {X_train_lemma_miss_tfidf.shape}")

=== Векторизация lemmatization + misspellings ===
Binary: (3092, 1122)
Counts: (3092, 1122)
TF-IDF: (3092, 1122)


**Stemming + stopwords**

In [20]:
# Для метода 'stemming + stopwords' делаем 3 вида векторизации

print("=== Векторизация stemming + stopwords ===")


# 1. Binary
bin_vec_stem_stop = CountVectorizer(
    binary=True,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_stem_stop_bin = bin_vec_stem_stop.fit_transform(X_train_stem_stop)
X_test_stem_stop_bin = bin_vec_stem_stop.transform(X_test_stem_stop)

print(f"Binary: {X_train_stem_stop_bin.shape}")


# 2. Counts
cnt_vec_stem_stop = CountVectorizer(
    binary=False,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_stem_stop_cnt = cnt_vec_stem_stop.fit_transform(X_train_stem_stop)
X_test_stem_stop_cnt = cnt_vec_stem_stop.transform(X_test_stem_stop)

print(f"Counts: {X_train_stem_stop_cnt.shape}")


# 3. TF-IDF
tfidf_vec_stem_stop = TfidfVectorizer(
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)

X_train_stem_stop_tfidf = tfidf_vec_stem_stop.fit_transform(X_train_stem_stop)
X_test_stem_stop_tfidf = tfidf_vec_stem_stop.transform(X_test_stem_stop)

print(f"TF-IDF: {X_train_stem_stop_tfidf.shape}")

=== Векторизация stemming + stopwords ===
Binary: (3092, 782)
Counts: (3092, 782)
TF-IDF: (3092, 782)


## **Similarity**

Поиск похожих твитов

In [21]:
# Для similarity используем весь датасет
texts_similarity = df['text'].drop_duplicates().reset_index(drop=True)

print("Всего твитов:", len(df))
print("Уникальных твитов для similarity:", len(texts_similarity))

Всего твитов: 3865
Уникальных твитов для similarity: 3451


In [22]:
preprocessing_functions = {

    'tokenization':
        lambda text: preprocess(text, method='tokenize'),

    'stemming':
        lambda text: preprocess(text, method='stem'),

    'lemmatization':
        lambda text: preprocess(text, method='lemma'),

    'stemming + misspellings':
        preprocess_stem_misspellings,

    'lemmatization + misspellings':
        preprocess_lemma_misspellings,

    'stemming + stopwords':
        lambda text: preprocess(text, method='stem_stop')
}

In [23]:
similarity_texts = {}

for name, func in preprocessing_functions.items():

    print("Обработка:", name)

    temp_df = pd.DataFrame({
        'original': texts_similarity
    })

    # применяем preprocessing
    temp_df['processed'] = temp_df['original'].apply(func)

    # убираем пустые строки
    temp_df = temp_df[
        temp_df['processed'].str.strip() != ''
    ]

    # убираем тексты, которые после preprocessing
    # стали полностью одинаковыми
    temp_df = temp_df.drop_duplicates(
        subset='processed'
    ).reset_index(drop=True)

    similarity_texts[name] = temp_df

    print("Осталось твитов:", len(temp_df))

Обработка: tokenization
Осталось твитов: 3408
Обработка: stemming
Осталось твитов: 3407
Обработка: lemmatization
Осталось твитов: 3407
Обработка: stemming + misspellings
Осталось твитов: 3407
Обработка: lemmatization + misspellings
Осталось твитов: 3407
Обработка: stemming + stopwords
Осталось твитов: 3331


In [24]:
def get_top_similar_pairs(temp_df, vectorizer_type, top_n=10):

    # Выбираем способ векторизации
    if vectorizer_type == 'binary':

        vectorizer = CountVectorizer(
            binary=True,
            min_df=1,
            ngram_range=(1, 2),
            token_pattern=r'(?u)\b\w+\b'
        )

    elif vectorizer_type == 'counts':

        vectorizer = CountVectorizer(
            binary=False,
            min_df=1,
            ngram_range=(1, 2),
            token_pattern=r'(?u)\b\w+\b'
        )

    elif vectorizer_type == 'tfidf':

        vectorizer = TfidfVectorizer(
            min_df=1,
            ngram_range=(1, 2),
            token_pattern=r'(?u)\b\w+\b'
        )

    else:
        raise ValueError(
            "vectorizer_type должен быть binary, counts или tfidf"
        )

    # Превращаем тексты в векторы
    X_vec = vectorizer.fit_transform(
        temp_df['processed']
    )

    # Сравниваем каждый твит с каждым
    similarity = cosine_similarity(X_vec)

    # Убираем A с A и повтор пары B-A, если уже существует A-B
    for i in range(similarity.shape[0]):
        similarity[i, :i + 1] = -1

    # Превращаем матрицу в один длинный массив
    flat = similarity.ravel()

    # Находим top_n максимальных значений
    top_indices = np.argpartition(
        flat,
        -top_n
    )[-top_n:]

    # Сортируем от самого похожего
    top_indices = top_indices[
        np.argsort(flat[top_indices])[::-1]
    ]

    # Возвращаем номера двух твитов
    rows, cols = np.unravel_index(
        top_indices,
        similarity.shape
    )

    # Формируем результат
    result = pd.DataFrame({
        'tweet_1': [
            temp_df['original'].iloc[i]
            for i in rows
        ],

        'tweet_2': [
            temp_df['original'].iloc[j]
            for j in cols
        ],

        'similarity': [
            similarity[i, j]
            for i, j in zip(rows, cols)
        ]
    })

    return result

In [25]:
vectorizer_types = [
    'binary',
    'counts',
    'tfidf'
]

similarity_results = []

for preprocessing_name, temp_df in similarity_texts.items():

    for vectorizer_type in vectorizer_types:

        print(
            preprocessing_name,
            "+",
            vectorizer_type
        )

        result = get_top_similar_pairs(
            temp_df,
            vectorizer_type,
            top_n=10
        )

        result.insert(
            0,
            'vectorizer',
            vectorizer_type
        )

        result.insert(
            0,
            'preprocessing',
            preprocessing_name
        )

        result.insert(
            2,
            'rank',
            range(1, 11)
        )

        similarity_results.append(result)

tokenization + binary
tokenization + counts
tokenization + tfidf
stemming + binary
stemming + counts
stemming + tfidf
lemmatization + binary
lemmatization + counts
lemmatization + tfidf
stemming + misspellings + binary
stemming + misspellings + counts
stemming + misspellings + tfidf
lemmatization + misspellings + binary
lemmatization + misspellings + counts
lemmatization + misspellings + tfidf
stemming + stopwords + binary
stemming + stopwords + counts
stemming + stopwords + tfidf


In [45]:
similarity_results_df = pd.concat(
    similarity_results,
    ignore_index=True
)

print(
    "Всего найдено пар:",
    len(similarity_results_df)
)

similarity_results_df.head(10)

Всего найдено пар: 180


,preprocessing,vectorizer,rank,tweet_1,tweet_2,similarity
0,tokenization,binary,1,Koalas are dying of thirst and it's all becau...,are dying of thirst and it's all because of u...,0.957427
1,tokenization,binary,2,Hi! We tried to call your number but got no re...,Hi! We tried to call your number but got no re...,0.953746
2,tokenization,binary,3,for the recent follow. Much appreciated happy ...,thanks for the recent follow. Much appreciated...,0.945905
3,tokenization,binary,4,Hi! We tried to call your number but got no re...,Hi! We tried to call your number but got no re...,0.945569
4,tokenization,binary,5,Share the love: thanks for being top new follo...,Share the love: thanks for being top new follo...,0.943564
5,tokenization,binary,6,Hey thanks for being top new followers this we...,Hey thanks for being top new followers this we...,0.940540
6,tokenization,binary,7,Share the love: thanks for being top new follo...,Share the love: thanks for being top new follo...,0.940540
7,tokenization,binary,8,Thanks for the recent follow Happy to connect ...,Thanks for the recent follow Happy to connect ...,0.928571
8,tokenization,binary,9,Thanks for the recent follow Happy to connect ...,Thanks for the recent follow Happy to connect ...,0.928571
9,tokenization,binary,10,Thanks for the recent follow Happy to connect ...,Thanks for the recent follow Happy to connect ...,0.928571


## **Machine Learning**

Собираем 18 подготовленных наборов и обучаем на них три модели

In [27]:
datasets = {

    # JUST TOKENIZATION
    ('tokenization', 'binary'):
        (X_train_tok_bin, X_test_tok_bin),

    ('tokenization', 'counts'):
        (X_train_tok_cnt, X_test_tok_cnt),

    ('tokenization', 'tfidf'):
        (X_train_tok_tfidf, X_test_tok_tfidf),


    # STEMMING
    ('stemming', 'binary'):
        (X_train_stem_bin, X_test_stem_bin),

    ('stemming', 'counts'):
        (X_train_stem_cnt, X_test_stem_cnt),

    ('stemming', 'tfidf'):
        (X_train_stem_tfidf, X_test_stem_tfidf),


    # LEMMATIZATION
    ('lemmatization', 'binary'):
        (X_train_lemma_bin, X_test_lemma_bin),

    ('lemmatization', 'counts'):
        (X_train_lemma_cnt, X_test_lemma_cnt),

    ('lemmatization', 'tfidf'):
        (X_train_lemma_tfidf, X_test_lemma_tfidf),


    # STEMMING + MISSPELLINGS
    ('stemming + misspellings', 'binary'):
        (X_train_stem_miss_bin, X_test_stem_miss_bin),

    ('stemming + misspellings', 'counts'):
        (X_train_stem_miss_cnt, X_test_stem_miss_cnt),

    ('stemming + misspellings', 'tfidf'):
        (X_train_stem_miss_tfidf, X_test_stem_miss_tfidf),


    # LEMMATIZATION + MISSPELLINGS
    ('lemmatization + misspellings', 'binary'):
        (X_train_lemma_miss_bin, X_test_lemma_miss_bin),

    ('lemmatization + misspellings', 'counts'):
        (X_train_lemma_miss_cnt, X_test_lemma_miss_cnt),

    ('lemmatization + misspellings', 'tfidf'):
        (X_train_lemma_miss_tfidf, X_test_lemma_miss_tfidf),


    # STEMMING + STOPWORDS
    ('stemming + stopwords', 'binary'):
        (X_train_stem_stop_bin, X_test_stem_stop_bin),

    ('stemming + stopwords', 'counts'):
        (X_train_stem_stop_cnt, X_test_stem_stop_cnt),

    ('stemming + stopwords', 'tfidf'):
        (X_train_stem_stop_tfidf, X_test_stem_stop_tfidf)
}

print("Количество датасетов:", len(datasets))

Количество датасетов: 18


In [28]:
ml_results = []
trained_models = {}

for (preprocessing_name, vectorizer_type), (Xtr, Xte) in datasets.items():

    models = {
        'LogisticRegression':
            LogisticRegression(
                max_iter=3000,
                random_state=42
            ),

        'LinearSVC':
            LinearSVC(dual='auto'),

        'MultinomialNB':
            MultinomialNB()
    }

    for model_name, model in models.items():

        model.fit(Xtr, y_train)

        y_pred = model.predict(Xte)

        accuracy = accuracy_score(
            y_test,
            y_pred
        )

        ml_results.append({
            'preprocessing': preprocessing_name,
            'vectorizer': vectorizer_type,
            'model': model_name,
            'accuracy': accuracy
        })

        trained_models[
            (
                preprocessing_name,
                vectorizer_type,
                model_name
            )
        ] = model

In [29]:
ml_results_df = pd.DataFrame(ml_results)

ml_results_df = ml_results_df.sort_values(
    by='accuracy',
    ascending=False
).reset_index(drop=True)

ml_results_df

,preprocessing,vectorizer,model,accuracy
0,tokenization,tfidf,LogisticRegression,0.892626
1,lemmatization + misspellings,tfidf,LogisticRegression,0.891332
2,tokenization,counts,LogisticRegression,0.890039
3,tokenization,tfidf,LinearSVC,0.890039
4,stemming + misspellings,counts,LogisticRegression,0.890039
5,stemming + misspellings,binary,LogisticRegression,0.888745
6,lemmatization + misspellings,binary,LogisticRegression,0.888745
7,lemmatization,tfidf,LogisticRegression,0.887451
8,lemmatization + misspellings,counts,LogisticRegression,0.886158
9,lemmatization,binary,LogisticRegression,0.884864


### **GridSearch and ROC-AUC**

Донастраиваем лучшую модель и считаем ROC-AUC.

In [30]:
param_grid = {
    'C': [0.1, 1, 10]
}

grid_lr = GridSearchCV(
    LogisticRegression(
        max_iter=3000,
        random_state=42
    ),
    param_grid=param_grid,
    cv=5,
    scoring='roc_auc_ovr'
)

In [31]:
grid_lr.fit(
    X_train_tok_tfidf,
    y_train
)

print("Лучшие параметры:", grid_lr.best_params_)
print("Лучший CV AUC:", grid_lr.best_score_)

Лучшие параметры: {'C': 1}
Лучший CV AUC: 0.9711675500244222


In [32]:
# Берём лучшую модель, найденную GridSearch
best_lr = grid_lr.best_estimator_

# Получаем вероятности принадлежности к каждому классу
y_test_proba = best_lr.predict_proba(
    X_test_tok_tfidf
)

# Считаем multiclass ROC-AUC на test
test_auc = roc_auc_score(
    y_test,
    y_test_proba,
    multi_class='ovr',
    average='macro'
)

print("Test ROC-AUC:", test_auc)

Test ROC-AUC: 0.9783857009238206


In [33]:
print("Лучшие параметры GridSearch:", grid_lr.best_params_)
print("CV ROC-AUC:", round(grid_lr.best_score_, 4))
print("Test ROC-AUC:", round(test_auc, 4))

print("\nОбязательное требование AUC > 0.832")

if test_auc > 0.832:
    print("Требование проекта выполнено")
else:
    print("Требование проекта не выполнено")

Лучшие параметры GridSearch: {'C': 1}
CV ROC-AUC: 0.9712
Test ROC-AUC: 0.9784

Обязательное требование AUC > 0.832
Требование проекта выполнено


In [34]:
final_result = pd.DataFrame({
    'preprocessing': ['tokenization'],
    'vectorizer': ['tfidf'],
    'model': ['LogisticRegression'],
    'best_params': [grid_lr.best_params_],
    'accuracy': [ml_results_df.iloc[0]['accuracy']],
    'cv_roc_auc': [grid_lr.best_score_],
    'test_roc_auc': [test_auc]
})

final_result

,preprocessing,vectorizer,model,best_params,accuracy,cv_roc_auc,test_roc_auc
0,tokenization,tfidf,LogisticRegression,{'C': 1},0.892626,0.971168,0.978386


In [35]:
best_by_model = (
    ml_results_df
    .sort_values('accuracy', ascending=False)
    .groupby('model', as_index=False)
    .first()
)

In [36]:
final_models = best_by_model.copy()

final_models['parameters'] = final_models['model'].map({
    'LogisticRegression': "C=1 (GridSearch), max_iter=3000",
    'LinearSVC': "C=1.0, dual='auto'",
    'MultinomialNB': "alpha=1.0, fit_prior=True"
})

final_models = final_models[
    [
        'model',
        'preprocessing',
        'vectorizer',
        'parameters',
        'accuracy'
    ]
]

final_models

,model,preprocessing,vectorizer,parameters,accuracy
0,LinearSVC,tokenization,tfidf,"C=1.0, dual='auto'",0.890039
1,LogisticRegression,tokenization,tfidf,"C=1 (GridSearch), max_iter=3000",0.892626
2,MultinomialNB,lemmatization + misspellings,counts,"alpha=1.0, fit_prior=True",0.875809


In [37]:
# Предсказания лучшей модели после GridSearch
best_predictions = best_lr.predict(
    X_test_tok_tfidf
)

print(
    classification_report(
        y_test,
        best_predictions,
        labels=[0, 1, 2],
        target_names=[
            'negative',
            'neutral',
            'positive'
        ]
    )
)

              precision    recall  f1-score   support

    negative       0.93      0.86      0.89       223
     neutral       0.85      0.98      0.91       313
    positive       0.93      0.81      0.86       237

    accuracy                           0.89       773
   macro avg       0.90      0.88      0.89       773
weighted avg       0.90      0.89      0.89       773



In [38]:
print("Количество результатов similarity:", len(similarity_results_df))

print("\nКоличество пар для каждой комбинации:")
print(
    similarity_results_df
    .groupby(['preprocessing', 'vectorizer'])
    .size()
)

Количество результатов similarity: 180

Количество пар для каждой комбинации:
preprocessing                 vectorizer
lemmatization                 binary        10
                              counts        10
                              tfidf         10
lemmatization + misspellings  binary        10
                              counts        10
                              tfidf         10
stemming                      binary        10
                              counts        10
                              tfidf         10
stemming + misspellings       binary        10
                              counts        10
                              tfidf         10
stemming + stopwords          binary        10
                              counts        10
                              tfidf         10
tokenization                  binary        10
                              counts        10
                              tfidf         10
dtype: int64


In [39]:
print("Количество ML-экспериментов:", len(ml_results_df))

print("\nКоличество моделей для каждой комбинации:")
print(
    ml_results_df
    .groupby(['preprocessing', 'vectorizer'])
    .size()
)

Количество ML-экспериментов: 54

Количество моделей для каждой комбинации:
preprocessing                 vectorizer
lemmatization                 binary        3
                              counts        3
                              tfidf         3
lemmatization + misspellings  binary        3
                              counts        3
                              tfidf         3
stemming                      binary        3
                              counts        3
                              tfidf         3
stemming + misspellings       binary        3
                              counts        3
                              tfidf         3
stemming + stopwords          binary        3
                              counts        3
                              tfidf         3
tokenization                  binary        3
                              counts        3
                              tfidf         3
dtype: int64
